In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from models import SimpleNet
import torch
import torch.nn as nn
import mlflow
import numpy as np
from torchmetrics.classification import BinaryAUROC
from imblearn.over_sampling import SMOTENC


c:\Users\wenyu\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print(f"Using CPU")
    device = torch.device("cpu")


Using GPU: NVIDIA GeForce GTX 1650


In [3]:
!nvidia-smi


Wed Sep 23 12:04:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.83                 Driver Version: 572.83         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8              7W /   35W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
df = pd.read_csv("data/train-cat-encoded.csv")
print("Training set size: ", df.shape)
y = df['Will_Buy_EV']
x = df.drop(columns=['Will_Buy_EV', 'id'])
print(x.shape)


Training set size:  (668665, 22)
(668665, 20)


In [5]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
subsetNum = 50000
subset_x = x.iloc[:subsetNum]
subset_y = y.iloc[:subsetNum]
print(subset_x.shape)


(50000, 20)


In [6]:
binary_features = [
    # 'Home_Charging_Possible', 
    'Subsidy_Available', 
    'City_Type_Urban', 'City_Type_Suburban', 
    # 'City_Type_Rural', 
    'Current_Car_Type_Sedan', 'Current_Car_Type_SUV', 
    # 'Current_Car_Type_Hatchback', 
    # 'Current_Car_Type_Truck', 
    # 'Gender_Male', 'Gender_Female', 'Gender_Other'
    ]

smote = SMOTENC(
    random_state=42,
    categorical_features=binary_features)
X_train_resampled, y_train_resampled = smote.fit_resample(subset_x, subset_y)
print(X_train_resampled.shape)


(82622, 20)


### Continous feature transformation
TODO
- Log transform
- standardisation
- min-max normalisation
- quantile binning (Daily_Commute_km)
- clipping/winsorizing

In [7]:
transformed_x = X_train_resampled.copy()
# min_max_col = []
log_transform = ['Annual_Income_USD']
standard_transform = [
    'Age', 'Environmental_Concern_Level', 
    'Number_of_Cars_Owned', 'Range_Anxiety_Level', 
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 
    'Daily_Commute_km'
    ]

# for col in min_max_col:
#     transformed_x[f"{col}_min_max"] = (transformed_x[col] - np.min(transformed_x[col])) / (np.max(transformed_x[col] - np.min(transformed_x[col])))
#     transformed_x = transformed_x.drop(columns=col)

for col in log_transform:
    transformed_x[f"{col}_log_trans"] = np.log1p(transformed_x[col])
    transformed_x = transformed_x.drop(columns=col)
    transformed_x[f"{col}_log_trans"] = (transformed_x[f"{col}_log_trans"] - np.mean(transformed_x[f"{col}_log_trans"])) / np.std(transformed_x[f"{col}_log_trans"])

for col in standard_transform:
    transformed_x[f"{col}_std_trans"] = (transformed_x[col] - np.mean(transformed_x[col])) / np.std(transformed_x[col])
    transformed_x = transformed_x.drop(columns=col)
print(transformed_x.shape)


(82622, 20)


In [8]:
x_tensor = torch.as_tensor(transformed_x.to_numpy(), dtype=torch.float32)
y_tensor = torch.as_tensor(y_train_resampled.to_numpy(), dtype=torch.float32)
n_sample, n_feature = x_tensor.shape
print(n_sample, n_feature)
print(y_tensor.shape)


82622 20
torch.Size([82622])


C:\Users\wenyu\AppData\Local\Temp\ipykernel_37816\3618563667.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  y_tensor = torch.as_tensor(y_train_resampled.to_numpy(), dtype=torch.float32)


In [9]:
# Training Parameters
params = {
    'lr': 0.005,
    'epoch': 5,
    'hidden_nodes': 10,
    'batch_size': 32,
    'training_sample': n_sample,
    'training_features': n_feature
}


In [10]:
model = SimpleNet(params['training_features'], 10, 2).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=params['lr'])
auc = BinaryAUROC()

for name, param in model.named_parameters():
    print(f"{name}: {param.dtype}")


layer1.weight: torch.float32
layer1.bias: torch.float32
layer2.weight: torch.float32
layer2.bias: torch.float32


In [11]:
mlflow.set_experiment("Neural Network Experiment")
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

bs = params['batch_size']

with mlflow.start_run() as run:
    mlflow.log_params(params)
    num_batches = 0

    for epoch in range(params['epoch']):
        epoch_batches = 0
        total_loss = 0
        
        for batch_idx in range(0, params['training_sample'], bs):
            feature_mat = x_tensor[batch_idx:batch_idx+bs].to(device)
            true_label = y_tensor[batch_idx:batch_idx+bs].to(device)

            #Forward pass
            optimiser.zero_grad()
            inputs = feature_mat.view(params['batch_size'], params['training_features'])

            score = model.forward(inputs)
            # pred_label = score[:, 1]
            pred_label = score[:, 1]

            # print(pred_label.shape)
            # print(true_label.shape)

            #Calculate metrics
            loss = loss_fn(pred_label, true_label)
            auc_score = auc(pred_label, true_label)
            total_loss += loss.item()
            # print(pred_label.min().item(), pred_label.max().item(), pred_label.std().item())
            print(f"{epoch}: {batch_idx}===Loss: {loss}===Auc: {auc_score}===Total Loss: {total_loss}")

            #Propagate backwards
            loss.backward()
            optimiser.step()

            #Log batch metrics
            num_batches += 1
            epoch_batches += 1
            batch_loss = total_loss / epoch_batches
            mlflow.log_metrics(
                {
                    'training_loss': loss.item(),
                    'training_auc_roc': auc_score.item(),
                    "avg_batch_loss": batch_loss
                }, 
                step=num_batches,
            )
            

        #Log epoch metrics
        # mlflow.log_metrics(
        #     {
        #         'loss': loss.item(),
        #         'auc_roc': auc_score.item(),
        #         "batch_loss": batch_loss
        #     }, 
        #     step=epoch * params['training_features'] + batch_idx,
        # )

        #Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f'checkpoint_{epoch}')

    model_info = mlflow.pytorch.log_model(model, name="final_model")  



            

2026/09/23 12:04:48 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/09/23 12:04:48 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


0: 0===Loss: 0.7466377019882202===Auc: 0.421875===Total Loss: 0.7466377019882202
0: 32===Loss: 0.7032898664474487===Auc: 0.5603864789009094===Total Loss: 1.449927568435669
0: 64===Loss: 0.7359622716903687===Auc: 0.43703705072402954===Total Loss: 2.1858898401260376
0: 96===Loss: 0.7422236204147339===Auc: 0.5999999642372131===Total Loss: 2.9281134605407715
0: 128===Loss: 0.7470670938491821===Auc: 0.2814815044403076===Total Loss: 3.6751805543899536
0: 160===Loss: 0.7168818712234497===Auc: 0.6000000238418579===Total Loss: 4.392062425613403
0: 192===Loss: 0.7288376688957214===Auc: 0.3571428656578064===Total Loss: 5.120900094509125
0: 224===Loss: 0.7193760871887207===Auc: 0.5166666507720947===Total Loss: 5.8402761816978455
0: 256===Loss: 0.7163652181625366===Auc: 0.4427083134651184===Total Loss: 6.556641399860382
0: 288===Loss: 0.7064242362976074===Auc: 0.64000004529953===Total Loss: 7.2630656361579895
0: 320===Loss: 0.7159932255744934===Auc: 0.37272727489471436===Total Loss: 7.9790588617324

c:\Users\wenyu\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


0: 17216===Loss: 0.3978319764137268===Auc: 0.7185185551643372===Total Loss: 280.4680318236351
0: 17248===Loss: 0.3569589853286743===Auc: 0.8303570747375488===Total Loss: 280.8249908089638
0: 17280===Loss: 0.3470088243484497===Auc: 0.8303571343421936===Total Loss: 281.1719996333122
0: 17312===Loss: 0.3668246269226074===Auc: 0.7767857313156128===Total Loss: 281.53882426023483
0: 17344===Loss: 0.511558473110199===Auc: 0.816425085067749===Total Loss: 282.05038273334503
0: 17376===Loss: 0.4084562063217163===Auc: 0.9102563858032227===Total Loss: 282.45883893966675
0: 17408===Loss: 0.3744548559188843===Auc: 0.714285671710968===Total Loss: 282.83329379558563
0: 17440===Loss: 0.5180088877677917===Auc: 0.5942857265472412===Total Loss: 283.3513026833534
0: 17472===Loss: 0.5013248324394226===Auc: 0.8072916269302368===Total Loss: 283.85262751579285
0: 17504===Loss: 0.5690142512321472===Auc: 0.748792290687561===Total Loss: 284.421641767025
0: 17536===Loss: 0.48422038555145264===Auc: 0.90104162693023

c:\Users\wenyu\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No negative samples in targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)


0: 50048===Loss: 1.166773796081543===Auc: 0.0===Total Loss: 647.4490452259779
0: 50080===Loss: 1.230142593383789===Auc: 0.0===Total Loss: 648.6791878193617
0: 50112===Loss: 1.1539294719696045===Auc: 0.0===Total Loss: 649.8331172913313
0: 50144===Loss: 1.0722519159317017===Auc: 0.0===Total Loss: 650.905369207263
0: 50176===Loss: 1.0815385580062866===Auc: 0.0===Total Loss: 651.9869077652693
0: 50208===Loss: 1.103206753730774===Auc: 0.0===Total Loss: 653.090114519
0: 50240===Loss: 1.1238045692443848===Auc: 0.0===Total Loss: 654.2139190882444
0: 50272===Loss: 1.1049177646636963===Auc: 0.0===Total Loss: 655.3188368529081
0: 50304===Loss: 1.0994340181350708===Auc: 0.0===Total Loss: 656.4182708710432
0: 50336===Loss: 0.9693025350570679===Auc: 0.0===Total Loss: 657.3875734061003
0: 50368===Loss: 0.9960505962371826===Auc: 0.0===Total Loss: 658.3836240023375
0: 50400===Loss: 1.1259430646896362===Auc: 0.0===Total Loss: 659.5095670670271
0: 50432===Loss: 0.995276689529419===Auc: 0.0===Total Loss: 

2026/09/23 12:07:51 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/09/23 12:07:51 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


KeyboardInterrupt: 